# Chapter 5: Multi-Token Prediction and FP8 Quantization

We have now established the core architectural pillars of the DeepSeek model: Multi-Head Latent Attention and Mixture-of-Experts. These innovations define *what* the model computes. Now, we turn our attention to an equally important topic that defines *how* these computations are performed with incredible efficiency.

This chapter covers two key techniques central to DeepSeek's training methodology:

1. **Multi-Token Prediction (MTP)** — A technique that provides stronger training signals by predicting multiple future tokens simultaneously
2. **FP8 Quantization** — An advanced quantization framework that trades precision for significant gains in speed and memory

This chapter is divided into two main parts. First, we will dive deep into MTP, understanding its motivation, advantages, and exactly how DeepSeek implemented their advanced, causal version of it. After mastering MTP, we will move to the second part, a deep dive into the FP8 Quantization framework.

## Required Libraries

Let's begin by importing the necessary Python libraries:

- `math`: For mathematical operations
- `typing.Optional`: For type hints with optional parameters
- `torch`: PyTorch library for tensor operations and neural networks
- `torch.nn`: Neural network modules and functions
- `torch.nn.functional`: Functional interface for neural network operations

In [ ]:
import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

---

# Part 1: Multi-Token Prediction (MTP)

## From Single-Token to Multi-Token Prediction

The entire training process for the language models we've discussed so far has been based on a single, simple objective: **Next-Token Prediction**. In the standard approach, the model's goal is to predict the single token that immediately follows each input token.

**Multi-Token Prediction** changes this fundamental objective. Instead of predicting only the single next token, the model is trained to predict **multiple future tokens at once**. For example, from the input token "is," it might predict the sequence "transforming," "the," "world."

### The Four Key Advantages of MTP

1. **Densification of training signals** — Each training sample contains much more information for the model to learn from, as it receives gradient signals across multiple future steps simultaneously.

2. **Improved data efficiency** — Since each training sample is more informative, the model achieves higher performance with the same amount of training data. The original MTP paper demonstrated this on coding benchmarks like MBPP and HumanEval.

3. **Better planning by prioritizing "choice points"** — The MTP loss implicitly assigns higher weights to critical tokens that significantly influence the future outcome, forcing the model to develop better internal representations for planning.

4. **Higher inference speed via speculative decoding** — MTP heads can serve as a "draft" model for speculative decoding, achieving up to 3x speedups on certain tasks.

## The DeepSeek MTP Architecture

While the original MTP paper from Meta proved the concept's effectiveness using **independent** output heads, DeepSeek recognized a key opportunity for improvement. Their implementation is designed to **sequentially** predict additional tokens and keep the complete causal chain at each prediction depth.

### Key Design Principles:

1. **Starting Point**: The Shared Transformer Trunk processes input tokens to produce the initial hidden states $h^0$
2. **Sequential Chain**: A series of MTP Modules are chained together, each producing a refined hidden state for the next
3. **Each MTP Module**:
   - Takes the hidden state from the previous step ($h^{k-1}_i$) and the embedding of the future token ($\text{Emb}(t_{i+k})$)
   - Normalizes both via RMSNorm, concatenates, and projects back to model dimension
   - Passes through a dedicated Transformer block
   - Outputs a refined hidden state that serves dual purpose: passed to next head AND projected to vocabulary logits

### The Key Equations:

**Equation 5.1** — Merge and project:
$$h'^k_i = M_k[\text{RMSNorm}(h^{k-1}_i) \; ; \; \text{RMSNorm}(\text{Emb}(t_{i+k}))]$$

**Equation 5.2** — Transformer refinement:
$$h^k_i = \text{TRM}_k(h'^k_i)$$

**Equation 5.3** — Token prediction:
$$P^k_{i+k+1} = \text{OutHead}(h^k_i)$$

The total loss is the sum of the main next-token prediction loss and the weighted average of all MTP losses:
$$\mathcal{L} = \mathcal{L}_{\text{main}} + \frac{\lambda}{D} \sum_{k=1}^{D} \mathcal{L}^k_{\text{MTP}}$$

## Implementing a Causal Multi-Token Prediction Module from Scratch

We will build the entire MTP system in stages:

1. **RMSNorm**: The normalization layer used throughout the DeepSeek architecture
2. **The MTP Module**: The core component that processes one step in the causal chain
3. **The Main Model**: A wrapper class that integrates the main Transformer trunk and the chain of MTP modules
4. **The Forward Pass & Loss**: The complete logic for sequential prediction and the combined loss calculation

### Listing 5.1: RMSNorm

We begin by defining RMSNorm (Root Mean Square Layer Normalization), the specific normalization layer used throughout the DeepSeek architecture. This is a foundational utility that ensures numerical stability during training.

In [ ]:
class RMSNorm(nn.Module):
    """
    Implements Root Mean Square Layer Normalization.
    """
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Calculate the inverse square root of the mean of squares
        norm_x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        # Apply the learnable weight
        return self.weight * norm_x

### Listing 5.2: The Causal MTP Module

Next, we build the core of the MTP architecture: the `DeepSeekMTPModule`. It contains a single, dedicated Transformer block and the necessary projection layers. Its purpose is to take the hidden state from the previous step and the embedding of the next token, and produce a refined hidden state for the next step in the causal chain.

This class is a direct implementation of the logic from Equations 5.1 and 5.2:

1. **Normalize** both the previous hidden state and the future token embedding via separate RMSNorm layers
2. **Concatenate** them to form a merged embedding of dimension $2d$
3. **Project** back down to dimension $d$ via the projection matrix $M_k$
4. **Refine** through a dedicated Transformer block to produce the new hidden state

In [ ]:
class DeepSeekMTPModule(nn.Module):
    def __init__(self, d_model: int, nhead: int, dim_feedforward: int, dropout: float = 0.0):
        super().__init__()
        self.d_model = d_model

        self.projection_matrix = nn.Linear(2 * d_model, d_model, bias=False)

        self.transformer_block = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )

        self.norm_hidden = RMSNorm(d_model)
        self.norm_embed = RMSNorm(d_model)

    def forward(self, h_prev: torch.Tensor, future_token_embeds: torch.Tensor) -> torch.Tensor:
        h_normed = self.norm_hidden(h_prev)
        embed_normed = self.norm_embed(future_token_embeds)

        concatenated = torch.cat([h_normed, embed_normed], dim=-1)
        h_prime = self.projection_matrix(concatenated)

        h_output = self.transformer_block(h_prime)

        return h_output